In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm

nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

text_data = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
print(f"Загружено {len(text_data.data)} текстов.")


Загружено 11314 текстов.


In [7]:
import spacy
from tqdm import tqdm

import os
os.system("pip install spacy")
os.system("python -m spacy download en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

# Лемматизация текста с использованием spaCy
processed_texts = []
for document in tqdm(text_data.data, desc="Лемматизация"):
    if not document.strip():  # Пропускаем пустые строки
        processed_texts.append("")
        continue

    doc = nlp(document.lower())
    lemmatized = " ".join([token.lemma_ for token in doc if token.is_alpha and not token.is_stop])
    processed_texts.append(lemmatized)

print(processed_texts[0])





Лемматизация: 100%|██████████| 11314/11314 [10:54<00:00, 17.28it/s]

wonder enlighten car see day door sport car look late early call bricklin door small addition bumper separate rest body know tellme model engine spec year production car history info funky look car e mail


In [10]:
from sklearn.feature_extraction.text import CountVectorizer

max_features = 2000
num_topics = 20
num_top_words = 10


vectorizer = CountVectorizer(lowercase=True,
                             stop_words='english',
                             analyzer='word',
                             max_df=0.8,
                             min_df=10,
                             max_features=max_features)

doc_term_matrix = vectorizer.fit_transform(lemmatized_corpus)

#Фичи
vocab = vectorizer.get_feature_names_out()

print(f"Размер матрицы: {doc_term_matrix.shape}")
print(vocab[:10])

alpha = 1.0
beta = 0.01






Размер матрицы: (3514, 2000)
['abc' 'ability' 'able' 'absolute' 'absolutely' 'abuse' 'accelerator'
 'accept' 'acceptable' 'access']


In [12]:
import numpy as np
from tqdm import tqdm

class TopicModelLDA:
    def __init__(self, n_topics=10, alpha=None, beta=None, n_iter=50):
        self.n_topics = n_topics
        self.alpha = alpha
        self.beta = beta
        self.n_iter = n_iter


        self.topic_word_count = None
        self.topic_count = None
        self.doc_topic_count = None

        self.is_fitted = False

    def fit(self, dt_matrix):
        n_docs, n_words = dt_matrix.shape

        if self.alpha is None:
            self.alpha = np.ones(self.n_topics)
        if self.beta is None:
            self.beta = np.ones(n_words)

        self.topic_count = np.zeros(self.n_topics)
        self.topic_word_count = np.zeros((self.n_topics, n_words))
        self.doc_topic_count = np.zeros((n_docs, self.n_topics))

        doc_indices, word_indices = dt_matrix.nonzero()
        topic_assignments = np.random.choice(self.n_topics, len(doc_indices))

        for doc, word, topic in zip(doc_indices, word_indices, topic_assignments):
            self.topic_count[topic] += 1
            self.topic_word_count[topic, word] += 1
            self.doc_topic_count[doc, topic] += 1

        for iteration in tqdm(range(self.n_iter), desc="Training LDA"):
            for idx in range(len(doc_indices)):
                current_doc = doc_indices[idx]
                current_word = word_indices[idx]
                current_topic = topic_assignments[idx]

                self.topic_count[current_topic] -= 1
                self.topic_word_count[current_topic, current_word] -= 1
                self.doc_topic_count[current_doc, current_topic] -= 1

                topic_probs = (self.doc_topic_count[current_doc, :] + self.alpha) * \
                              (self.topic_word_count[:, current_word] + self.beta[current_word]) / \
                              (self.topic_count + self.beta.sum())

                topic_probs /= topic_probs.sum()

                new_topic = np.random.choice(self.n_topics, p=topic_probs)

                topic_assignments[idx] = new_topic
                self.topic_count[new_topic] += 1
                self.topic_word_count[new_topic, current_word] += 1
                self.doc_topic_count[current_doc, new_topic] += 1

        self.is_fitted = True
        return self

    def get_topic_word_matrix(self):
        if not self.is_fitted:
            raise ValueError("Модель не обучена. Вызовите fit() перед использованием.")
        return self.topic_word_count




In [14]:

train = vectorizer.fit_transform(lemmatized_corpus)
components = 20
max_iter = 50
lda_model = TopicModelLDA(n_topics=components, n_iter=max_iter)
lda_model.fit(train)

print("Обучение завершено!")


Training LDA: 100%|██████████| 50/50 [05:07<00:00,  6.14s/it]

Обучение завершено!


In [18]:
top_words = 10

result = np.argsort(lda_model.get_topic_word_matrix(), axis=1)[:, -top_words:]

for topic_idx in range(components):
    topic_words_matrix = np.zeros((1, train.shape[1]))
    for word_idx in result[topic_idx]:
        topic_words_matrix[0, word_idx] = 1
    topic_words = vectorizer.inverse_transform(topic_words_matrix)[0]
    print("Tag {}: {}".format(topic_idx + 1, ", ".join(topic_words)))



Tag 1: address, available, check, follow, include, information, list, post, read, send
Tag 2: attack, government, israel, kill, live, people, state, war, world, year
Tag 3: card, disk, drive, pc, problem, run, software, use, window, windows
Tag 4: buy, car, cost, drive, good, like, new, sell, want, work
Tag 5: game, good, play, player, second, team, think, way, win, year
Tag 6: end, good, know, mean, need, tell, thing, think, time, want
Tag 7: change, come, guess, know, like, say, thing, think, try, year
Tag 8: actually, good, know, like, people, right, tell, think, time, try
Tag 9: believe, christian, fact, god, man, mean, people, point, reason, say
Tag 10: good, happen, know, like, long, number, open, try, way, work
Tag 11: case, come, consider, course, hand, make, point, sure, think, way
Tag 12: able, base, build, change, high, large, level, new, point, year
Tag 13: big, make, mean, place, point, question, start, time, try, way
Tag 14: appreciate, help, hi, know, like, look, mail, n